In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# Encoder
  The encoder network takes in the data `input_dim`, processes it through a few fully connected layers, and outputs the mean ($mu$) and log-variance (log_var) of the latent distribution.

In [ ]:
# Define the Encoder network
class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Encoder, self).__init__()
        # Define a simple feed-forward network for the encoder
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3_mu = nn.Linear(256, latent_dim)  # Mean of the latent variable
        self.fc3_logvar = nn.Linear(256, latent_dim)  # Log variance of the latent variable

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        mu = self.fc3_mu(x)
        logvar = self.fc3_logvar(x)
        return mu, logvar

# Decoder

In [ ]:
# Define the Decoder network
class Decoder(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super(Decoder, self).__init__()
        # Define a simple feed-forward network for the decoder
        self.fc1 = nn.Linear(latent_dim, 256)
        self.fc2 = nn.Linear(256, 512)
        self.fc3 = nn.Linear(512, output_dim)

    def forward(self, z):
        z = F.relu(self.fc1(z))
        z = F.relu(self.fc2(z))
        reconstruction = torch.sigmoid(self.fc3(z))  # Sigmoid for binary outputs
        return reconstruction

# Reparameterization Trick

In [ ]:
# Reparameterization trick
def reparameterize(mu, logvar):
    std = torch.exp(0.5*logvar)
    eps = torch.randn_like(std)
    z = mu + eps*std
    return z

# VAE Model

In [ ]:
# VAE Model
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim, output_dim):
        super(VAE, self).__init__()
        self.encoder = Encoder(input_dim, latent_dim)
        self.decoder = Decoder(latent_dim, output_dim)

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = reparameterize(mu, logvar)
        reconstructed_x = self.decoder(z)
        return reconstructed_x, mu, logvar

    def loss_function(self, recon_x, x, mu, logvar):
        # Reconstruction loss (binary cross-entropy)
        BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')

        # KL divergence loss
        # Compute the KL divergence between the learned distribution and the prior (N(0, I))
        # Use the formula: KL(q(z|x) || p(z)) = 0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
        # where mu is the mean and logvar is the log variance
        # sigma^2 = exp(logvar)
        # Here we calculate the negative log-likelihood in a variational autoencoder
        # for an isotropic Gaussian distribution
        KL = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

        # The total loss is the sum of the reconstruction loss and KL divergence
        return BCE + KL

# Hyperparameters
input_dim = 64  # Adjust this depending on your EEG data
latent_dim = 10
output_dim = 64  # Same as input if reconstructing input (adjust accordingly)

# Create the VAE model
model = VAE(input_dim=input_dim, latent_dim=latent_dim, output_dim=output_dim)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3)


# Train

In [ ]:
# Training loop (simplified)
def train(model, train_loader, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        train_loss = 0
        for batch_idx, (data, _) in enumerate(train_loader):
            optimizer.zero_grad()
            recon_batch, mu, logvar = model(data)
            loss = model.loss_function(recon_batch, data, mu, logvar)
            loss.backward()
            train_loss += loss.item()
            optimizer.step()

        print(f'Epoch {epoch+1}, Loss: {train_loss / len(train_loader.dataset):.4f}')

In [ ]:
# Example usage with dummy dataset (replace with your EEG data)
class DummyDataset(Dataset):
    def __init__(self, num_samples, input_dim):
        self.data = torch.randn(num_samples, input_dim)  # Replace with your EEG data
        self.labels = torch.zeros(num_samples)  # Dummy labels, replace if necessary

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Instantiate dataset and data loader
dataset = DummyDataset(num_samples=1000, input_dim=input_dim)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

# Train the model
train(model, train_loader, optimizer)